# Task 2 - Expression model - Google Colab

**Course:** DLBAIPEAI - Project Edge AI  
**Student:** W. Pretorius  
**Platform:** Google Colab

This notebook is the Google Colab version of the expression model for **Task 2**. It trains an EfficientNetV2-B0 model to recognise seven facial expressions.

The classes are Angry, Disgust, Fear, Happy, Sad, Surprise and Neutral.

This version keeps the same group-aware data split, recall checks, balanced-accuracy model selection and TensorFlow Lite checks as the desktop notebook.

Before running the notebook, choose **Runtime → Change runtime type → GPU** in Google Colab.

In [ ]:
# cell:folder
import os
import sys
from pathlib import Path
from datetime import datetime

PROJECT_DIR = Path("/content/project_edge_ai")
PROJECT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TASK = "expression"
NOTEBOOK_NAME = "03_Expression_Model_Colab"

OUTPUT_DIR = (
    PROJECT_DIR
    / "results"
    / TASK
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["KERAS_HOME"] = str(
    PROJECT_DIR
    / "datasets"
    / "keras_cache"
)

os.environ["KAGGLEHUB_CACHE"] = str(
    PROJECT_DIR
    / "datasets"
    / "kagglehub_cache"
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("Python:", sys.version.split()[0])
print("Project folder:", PROJECT_DIR)
print("Results folder:", OUTPUT_DIR)

## 2. Install the extra libraries

In this cell we install the small helper libraries used by the notebook. TensorFlow itself comes from the Colab runtime.

In [ ]:
# cell:install
import subprocess
import sys
from importlib import metadata

# Colab already supplies TensorFlow and GPU support.
# Only small helper packages are installed here.
packages = {
    "kagglehub": "0.3.13",
    "tqdm": "4.67.1",
    "reportlab": "4.3.1",
    "nbformat": "5.10.4",
}

missing = []

for package, version in packages.items():
    try:
        installed = metadata.version(
            package
        )
    except metadata.PackageNotFoundError:
        installed = None

    if installed != version:
        missing.append(
            f"{package}=={version}"
        )

if missing:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            *missing,
        ]
    )

print("Colab helper libraries are ready.")

## 3. Keep the outputs for the PDF

In this cell we start recording later cell outputs for the final PDF. The code is kept here, not in a separate file. Setup-cell text is included, but its earlier output is not recorded.


In [ ]:
# cell:record
# This small tool stays inside the notebook. No helper file is needed.
import io
import re
import sys
import copy
import json

if "pdf_recorder" in globals():
    pdf_recorder.close()

class LiveNotebook:
    def __init__(self):
        self.ip = get_ipython()
        self.records = {}
        self.current = None
        self.stdout, self.stderr = sys.stdout, sys.stderr
        self.publish = self.ip.display_pub.publish
        self.format_data = self.ip.displayhook.write_format_data
        self.ip.display_pub.publish = self.capture_display
        self.ip.displayhook.write_format_data = self.capture_result
        self.ip.events.register("pre_run_cell", self.before)
        self.ip.events.register("post_run_cell", self.after)

    def before(self, info):
        self.restore()
        key = info.raw_cell.splitlines()[0].strip()
        self.current = key if key.startswith("# cell:") else None
        if self.current:
            self.records[self.current] = {
                "source": info.raw_cell,
                "execution_count": self.ip.execution_count,
                "outputs": [],
            }
            recorder = self
            class Tee:
                def __init__(self, original, name):
                    self.original, self.name = original, name
                def write(self, text):
                    outputs = recorder.records[recorder.current]["outputs"]
                    if outputs and outputs[-1].get("name") == self.name:
                        outputs[-1]["text"] += text
                    else:
                        outputs.append({"output_type": "stream", "name": self.name,
                                        "text": text})
                    return self.original.write(text)
                def flush(self):
                    return self.original.flush()
                def __getattr__(self, name):
                    return getattr(self.original, name)
            sys.stdout, sys.stderr = Tee(self.stdout, "stdout"), Tee(self.stderr, "stderr")

    def capture_display(self, data, metadata=None, **kwargs):
        if self.current:
            self.records[self.current]["outputs"].append({
                "output_type": "display_data", "data": copy.deepcopy(data),
                "metadata": metadata or {},
            })
        return self.publish(data=data, metadata=metadata, **kwargs)

    def capture_result(self, data, metadata=None):
        if self.current:
            self.records[self.current]["outputs"].append({
                "output_type": "execute_result", "data": copy.deepcopy(data),
                "metadata": metadata or {}, "execution_count": self.ip.execution_count,
            })
        return self.format_data(data, metadata)

    def restore(self):
        sys.stdout, sys.stderr = self.stdout, self.stderr

    def after(self, result):
        if self.current:
            error = result.error_before_exec or result.error_in_exec
            if error:
                self.records[self.current]["outputs"].append({
                    "output_type": "stream", "name": "stderr",
                    "text": f"{type(error).__name__}: {error}",
                })
        self.restore()
        self.current = None

    def close(self):
        self.restore()
        self.ip.display_pub.publish = self.publish
        self.ip.displayhook.write_format_data = self.format_data
        for event, function in [("pre_run_cell", self.before), ("post_run_cell", self.after)]:
            if function in self.ip.events.callbacks[event]:
                self.ip.events.unregister(event, function)

pdf_recorder = LiveNotebook()
print("Later cell outputs will be included in the PDF.")

## 4. Check that the Colab GPU can train

In this cell we check that Colab can see a GPU and run a real training calculation. We also enable mixed precision to make EfficientNetV2 training faster.

In [ ]:
# cell:imports
import time
import hashlib
import zipfile
import shutil
import platform
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import kagglehub
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from IPython.display import display, Markdown

get_ipython().run_line_magic(
    "matplotlib",
    "inline",
)

pd.set_option(
    "display.max_columns",
    12,
)

pd.set_option(
    "display.width",
    120,
)

print("TensorFlow:", tf.__version__)
print("System:", platform.platform())

gpus = tf.config.list_physical_devices(
    "GPU"
)

if not gpus:
    raise RuntimeError(
        "No GPU was found. In Colab choose Runtime → Change runtime type → GPU, "
        "then run the notebook again."
    )

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass

GPU_NAME = tf.config.experimental.get_device_details(
    gpus[0]
).get(
    "device_name",
    str(gpus[0]),
)

print("GPU found:", GPU_NAME)

# Check a real forward and backward calculation.
previous_placement = tf.config.get_soft_device_placement()
tf.config.set_soft_device_placement(
    False
)

try:
    with tf.device("/GPU:0"):
        probe_images = tf.random.normal(
            (2, 32, 32, 3)
        )

        probe_weights = tf.Variable(
            tf.random.normal(
                (3, 3, 3, 8)
            )
        )

        with tf.GradientTape() as tape:
            probe_output = tf.nn.conv2d(
                probe_images,
                probe_weights,
                strides=1,
                padding="SAME",
            )

            probe_loss = tf.reduce_mean(
                tf.square(
                    probe_output
                )
            )

        probe_gradient = tape.gradient(
            probe_loss,
            probe_weights,
        )

    if probe_gradient is None:
        raise RuntimeError(
            "The GPU did not calculate a gradient."
        )

    if "GPU" not in probe_output.device.upper():
        raise RuntimeError(
            "The test calculation did not run on the GPU."
        )

    if not np.isfinite(
        probe_gradient.numpy()
    ).all():
        raise RuntimeError(
            "The GPU gradient contains invalid values."
        )

finally:
    tf.config.set_soft_device_placement(
        previous_placement
    )

GPU_READY = True

# EfficientNetV2 is much faster with mixed precision on Colab GPUs.
tf.keras.mixed_precision.set_global_policy(
    "mixed_float16"
)

print(
    "Precision policy:",
    tf.keras.mixed_precision.global_policy().name,
)

print(
    "GPU forward and backward test: PASSED"
)

### Expression in-the-Wild dataset

For this model we use **ExpW**, or Expression in-the-Wild. It contains about 91,793 manually labelled faces with seven expression classes.

The dataset is downloaded from Kaggle inside this notebook. The original ExpW research was published by Zhang et al. The Kaggle mirror used here lists an MIT licence.

The seven labels are angry, disgust, fear, happy, sad, surprise and neutral.

## 5. Find or download the ExpW dataset

In this cell we check the Colab Kaggle cache first. If the ExpW labels and images are already present in this runtime, we reuse them. Kaggle is only used when a dataset is missing.

In [ ]:
# cell:dataset
LABEL_HANDLE = "nguhaduong/expression-in-the-wild-expw-dataset"
IMAGE_HANDLE = "minhtmnguyntrn/origin-expw"

KAGGLE_CACHE = Path(
    os.environ["KAGGLEHUB_CACHE"]
)
KAGGLE_CACHE.mkdir(parents=True, exist_ok=True)

def cached_dataset(owner, slug, check_function):
    """Return the newest usable cached version, or None."""
    versions_dir = KAGGLE_CACHE / "datasets" / owner / slug / "versions"

    if not versions_dir.is_dir():
        return None

    versions = sorted(
        [p for p in versions_dir.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    for version in versions:
        try:
            if check_function(version):
                return version.resolve()
        except Exception:
            pass

    return None


def has_expw_labels(root):
    return any(root.rglob("label.lst"))


def find_image_root(root):
    # The image Kaggle dataset normally contains an "origin" folder.
    origin_folders = [p for p in root.rglob("origin") if p.is_dir()]

    search_roots = origin_folders if origin_folders else [root]

    for candidate in search_roots:
        try:
            first_image = next(
                p for p in candidate.rglob("*")
                if p.is_file()
                and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
            )
            return first_image.parent if first_image.parent == candidate else candidate
        except StopIteration:
            continue

    return None


def has_expw_images(root):
    return find_image_root(root) is not None


# -------------------------
# Labels
# -------------------------
LABEL_DATA_DIR = cached_dataset(
    "nguhaduong",
    "expression-in-the-wild-expw-dataset",
    has_expw_labels,
)

if LABEL_DATA_DIR is None:
    print("ExpW label dataset was not found in the cache.")
    print("Downloading the labels from Kaggle...")
    LABEL_DATA_DIR = Path(
        kagglehub.dataset_download(LABEL_HANDLE)
    ).resolve()

    if not has_expw_labels(LABEL_DATA_DIR):
        raise FileNotFoundError(
            "The label dataset downloaded, but label.lst was not found."
        )
else:
    print("Verified cached ExpW labels.")


# -------------------------
# Original images
# -------------------------
IMAGE_DATA_DIR = cached_dataset(
    "minhtmnguyntrn",
    "origin-expw",
    has_expw_images,
)

if IMAGE_DATA_DIR is None:
    print("\nThe original ExpW images were not found in the cache.")
    print("Downloading the image archive from Kaggle.")
    print("This is a large download, about 8.4 GB on the first run.")

    IMAGE_DATA_DIR = Path(
        kagglehub.dataset_download(IMAGE_HANDLE)
    ).resolve()

    if not has_expw_images(IMAGE_DATA_DIR):
        raise FileNotFoundError(
            "The image dataset downloaded, but no JPG/PNG images were found."
        )
else:
    print("Verified cached ExpW original images.")


IMAGE_ROOT = find_image_root(IMAGE_DATA_DIR)

if IMAGE_ROOT is None:
    raise FileNotFoundError("Could not find the ExpW image folder.")

LABEL_FILE = max(
    LABEL_DATA_DIR.rglob("label.lst"),
    key=lambda p: p.stat().st_size,
)

print("\nLabel file:")
print(LABEL_FILE)

print("\nImage folder:")
print(IMAGE_ROOT)

# Count the image files once so we can verify the dataset.
image_count = sum(
    1
    for p in IMAGE_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

print("\nOriginal image files found:", image_count)

if image_count < 90000:
    raise RuntimeError(
        "The ExpW image archive looks incomplete. "
        f"Only {image_count} images were found."
    )

print("\nDataset check passed.")

## 6. Read the expression labels

In this cell we connect each ExpW label to its original image. We also keep the face box so the next steps can crop the correct face from the photo.

In [ ]:
# cell:labels
EXPW_NAMES = {
    0: "Angry",
    1: "Disgust",
    2: "Fear",
    3: "Happy",
    4: "Sad",
    5: "Surprise",
    6: "Neutral",
}

# Build a file-name index for the original ExpW images.
print("Indexing ExpW image files...")

image_paths = [
    p.resolve()
    for p in IMAGE_ROOT.rglob("*")
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

by_name = {}

for path in tqdm(image_paths, desc="Indexing images"):
    by_name.setdefault(path.name, []).append(path)

print("Unique image names indexed:", len(by_name))

rows = []
missing = []
ambiguous = []
bad_lines = 0

with LABEL_FILE.open(
    "r",
    encoding="utf-8-sig",
    errors="replace",
) as stream:

    for line in tqdm(
        stream,
        total=91793,
        desc="Reading ExpW labels",
    ):
        parts = line.strip().split()

        if len(parts) < 8:
            bad_lines += 1
            continue

        image_name = parts[0]

        try:
            face_id = int(float(parts[1]))
            top = float(parts[2])
            left = float(parts[3])
            right = float(parts[4])
            bottom = float(parts[5])
            confidence = float(parts[6])
            label = int(float(parts[7]))

        except ValueError:
            bad_lines += 1
            continue

        if label not in EXPW_NAMES:
            continue

        matches = by_name.get(image_name, [])

        if len(matches) == 0:
            missing.append(image_name)
            continue

        if len(matches) > 1:
            ambiguous.append(image_name)

        image_path = matches[0]

        width = right - left
        height = bottom - top

        rows.append({
            "path": str(image_path),
            "expression": label,
            "face_id": face_id,
            "face_x": left,
            "face_y": top,
            "face_width": width,
            "face_height": height,
            "face_confidence": confidence,
        })


raw = pd.DataFrame(rows)

print("\nLabel file:", LABEL_FILE)
print("Labelled faces loaded:", len(raw))
print("Missing image rows:", len(missing))
print("Ambiguous image names:", len(ambiguous))
print("Bad label lines:", bad_lines)

# Save a small diagnostic file only when something is missing.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if missing:
    pd.DataFrame(
        {"missing_image_name": sorted(set(missing))}
    ).to_csv(
        OUTPUT_DIR / "missing_expw_images.csv",
        index=False,
    )

if ambiguous:
    pd.DataFrame(
        {"ambiguous_image_name": sorted(set(ambiguous))}
    ).to_csv(
        OUTPUT_DIR / "ambiguous_expw_images.csv",
        index=False,
    )

if raw.empty:
    raise ValueError(
        "No labelled ExpW images were matched. "
        "Check that both Kaggle datasets were downloaded."
    )

# Nearly all 91,793 labelled faces should be available.
match_rate = len(raw) / 91793

print("Matched label rate:", f"{100 * match_rate:.2f}%")

if match_rate < 0.95:
    raise RuntimeError(
        "Too many ExpW labels could not be matched to images. "
        "The image archive may be incomplete or from a different dataset version."
    )

raw["expression"] = pd.to_numeric(
    raw["expression"],
    errors="raise",
).astype(int)

if not raw["expression"].between(0, 6).all():
    raise ValueError(
        "ExpW expression labels must be between 0 and 6."
    )

print("\nExpression counts:")

counts = (
    raw["expression"]
    .value_counts()
    .sort_index()
)

display(
    pd.DataFrame({
        "label_id": counts.index,
        "expression": [
            EXPW_NAMES[i]
            for i in counts.index
        ],
        "images": counts.values,
    })
)

print("\nTotal labelled faces:", len(raw))

## 7. Choose the training settings

In this cell we choose the image size, batch size and training length. These are starting choices. Validation data can guide changes; final holdout data must not.


In [ ]:
# cell:settings
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 25

CLASS_NAMES = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Sad",
    "Surprise",
    "Neutral",
]
N_CLASSES = len(CLASS_NAMES)
TARGET_COLUMN = "expression"

# Set this to True only when testing the full pipeline quickly.
# Keep it False for the final experiment.
FAST_MODE = False

if FAST_MODE:
    HEAD_EPOCHS = 2
    FINE_TUNE_EPOCHS = 3
    print(
        "FAST_MODE is ON. This is a quick pipeline test, not the final experiment."
    )
else:
    print(
        "FAST_MODE is OFF. Full training settings are active."
    )

tf.keras.utils.set_random_seed(SEED)

raw["label"] = raw["expression"].astype(int)

if raw["label"].isna().any() or not raw["label"].between(0, N_CLASSES - 1).all():
    raise ValueError("Some labels do not match the seven ExpW classes.")

raw["label"] = raw["label"].astype(int)

print("Class order:", dict(enumerate(CLASS_NAMES)))

# These rules are set before model training and before seeing conversion results.
MOBILE_IMAGES_PER_CLASS = 64

FP32_LIMITS = {
    "max_abs": 0.02,
    "mean_abs": 0.002,
    "min_agreement": 0.99,
    "max_balanced_accuracy_drop": 0.01,
    "max_class_recall_drop": 0.03,
}

INT8_LIMITS = {
    "max_abs": 0.15,
    "mean_abs": 0.03,
    "min_agreement": 0.96,
    "max_balanced_accuracy_drop": 0.03,
    "max_class_recall_drop": 0.05,
}

## 8. Use the full ExpW dataset

In this cell we keep all usable ExpW images. The dataset is already close to our target size, so we do not throw away good training images.

In [ ]:
# cell:sample
print("Labelled ExpW faces before image checks:", len(raw))
print("We keep the full dataset and remove only bad files or exact repeats later.")

class_table = (
    raw["label"]
    .value_counts()
    .sort_index()
    .rename("images")
    .to_frame()
)
class_table["expression"] = [CLASS_NAMES[i] for i in class_table.index]
display(class_table[["expression", "images"]])

## 9. Check the images

In this cell we open each image and check its face crop. We record bad files instead of hiding them. The check can take a few minutes.


In [ ]:
# cell:audit
BOX_COLUMNS = ["face_x", "face_y", "face_width", "face_height"]

def open_face(path, box):
    with Image.open(path) as original:
        original.load()
        image = original.convert("RGB")
    x, y, width, height = map(float, box)
    if not (x == y == width == height == -1):
        if not np.isfinite([x, y, width, height]).all() or width <= 0 or height <= 0:
            raise ValueError("Invalid face box.")
        bounds = (max(0, int(x)), max(0, int(y)),
                  min(image.width, int(x + width)), min(image.height, int(y + height)))
        if bounds[2] <= bounds[0] or bounds[3] <= bounds[1]:
            raise ValueError("Face box is outside the image.")
        image = image.crop(bounds)
    if min(image.size) < 16:
        raise ValueError("Face crop is smaller than 16 pixels.")
    return image

def check_image(row):
    try:
        image = open_face(row["path"], [row[column] for column in BOX_COLUMNS])
        digest = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
        return {"sha256": digest, "image_error": ""}
    except Exception as error:
        return {"sha256": "", "image_error": str(error)}

with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 1)) as pool:
    checks = list(tqdm(pool.map(check_image, raw.to_dict("records")),
                       total=len(raw), desc="Checking face images"))
checked = pd.concat([raw.reset_index(drop=True), pd.DataFrame(checks)], axis=1)

# Make sure the results folder still exists before saving.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

image_check_file = OUTPUT_DIR / "image_checks.csv"
checked.to_csv(image_check_file, index=False)

print("Image check saved to:", image_check_file)
print("Unreadable or invalid images:", int(checked["image_error"].ne("").sum()))
clean_data = checked.loc[checked["image_error"].eq("")].copy()
if clean_data.empty:
    raise ValueError("No usable face images were found.")

## 10. Remove exact repeats

In this cell we remove exact duplicate face images and any exact image that has conflicting labels. Different photos of the same person may still remain.

In [ ]:
# cell:duplicates
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
label_columns = ["label"]

conflicts = clean_data.groupby("sha256")[label_columns].nunique().max(axis=1)
conflict_hashes = set(conflicts[conflicts > 1].index)

conflicting = clean_data["sha256"].isin(conflict_hashes)
print("Images with conflicting labels:", int(conflicting.sum()))

clean_data = clean_data.loc[~conflicting].copy()
clean_data = clean_data.sort_values("path")

repeats = clean_data.duplicated("sha256", keep="first")
clean_data.loc[repeats].to_csv(OUTPUT_DIR / "duplicate_images.csv", index=False)

print("Exact repeated images removed:", int(repeats.sum()))

clean_data = clean_data.loc[~repeats].copy().reset_index(drop=True)

## 11. Separate training, validation and testing

In this cell we make about 80% training, 10% validation and 10% holdout data. Faces from the same original photograph always stay together in one split.

In [ ]:
# cell:split
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if clean_data["label"].value_counts().min() < 10:
    raise ValueError("A class has too few images for a reliable split.")

# ExpW can contain more than one labelled face from the same photograph.
# Every row with the same original image path must stay in the same split.
groups = clean_data["path"].astype(str).to_numpy()
labels_for_split = clean_data["label"].to_numpy(dtype=np.int32)

splitter = StratifiedGroupKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED,
)

fold_id = np.full(len(clean_data), -1, dtype=np.int32)

for fold, (_, held_index) in enumerate(
    splitter.split(clean_data, labels_for_split, groups)
):
    fold_id[held_index] = fold

if (fold_id < 0).any():
    raise RuntimeError("Some ExpW rows were not assigned to a split.")

test_df = clean_data.loc[fold_id == 0].copy()
val_df = clean_data.loc[fold_id == 1].copy()
train_df = clean_data.loc[fold_id >= 2].copy()

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

frames = {
    "train": train_df,
    "validation": val_df,
    "holdout": test_df,
}

for name, frame in frames.items():
    if set(frame["label"]) != set(range(N_CLASSES)):
        raise ValueError(
            f"The {name} set does not contain every expression class."
        )

    frame.to_csv(
        OUTPUT_DIR / f"{name}_images.csv",
        index=False,
    )

# Exact image data and original photographs must not cross splits.
for first_name, second_name in [
    ("train", "validation"),
    ("train", "holdout"),
    ("validation", "holdout"),
]:
    first = frames[first_name]
    second = frames[second_name]

    if not set(first["sha256"]).isdisjoint(set(second["sha256"])):
        raise RuntimeError(
            f"Exact duplicate leakage between {first_name} and {second_name}."
        )

    if not set(first["path"]).isdisjoint(set(second["path"])):
        raise RuntimeError(
            f"Original photograph leakage between {first_name} and {second_name}."
        )

print({
    name: len(frame)
    for name, frame in frames.items()
})

print(
    "Training percentage:",
    round(100 * len(train_df) / len(clean_data), 2),
)
print(
    "Validation percentage:",
    round(100 * len(val_df) / len(clean_data), 2),
)
print(
    "Holdout percentage:",
    round(100 * len(test_df) / len(clean_data), 2),
)

print("Group leakage check: PASSED")

# Optional small subset for a quick Colab pipeline test.
# This is applied only after the group-safe split.
if FAST_MODE:
    def quick_subset(frame, per_class):
        parts = []

        for label, group in frame.groupby(
            "label"
        ):
            parts.append(
                group.sample(
                    min(
                        per_class,
                        len(group),
                    ),
                    random_state=(
                        SEED
                        + int(label)
                    ),
                )
            )

        return pd.concat(
            parts,
            ignore_index=True,
        )

    train_df = quick_subset(
        train_df,
        1500,
    )

    val_df = quick_subset(
        val_df,
        250,
    )

    test_df = quick_subset(
        test_df,
        250,
    )

    frames = {
        "train": train_df,
        "validation": val_df,
        "holdout": test_df,
    }

    print(
        "FAST_MODE subset sizes:",
        {
            name: len(frame)
            for name, frame in frames.items()
        },
    )

## 12. Count each class

In this cell we count the images in each class. This shows whether some classes have much less training data.


In [ ]:
# cell:counts
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
counts = pd.DataFrame({name: frame["label"].value_counts().reindex(range(N_CLASSES), fill_value=0)
                       for name, frame in frames.items()})
counts.index = CLASS_NAMES
display(counts)
counts.to_csv(OUTPUT_DIR / "class_counts.csv")
fig, ax = plt.subplots(figsize=(9, 4))
counts["train"].plot.bar(ax=ax)
ax.set(title="Training images per class", xlabel="Class", ylabel="Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "class_counts.png", dpi=150)
plt.show()

## 13. Look at a few training images

In this cell we inspect a few face crops and labels. Keep these research images private. Set SHOW_FACE_IMAGES to False before making a copy that must not contain dataset photos.


In [ ]:
# cell:samples
SHOW_FACE_IMAGES = True
if SHOW_FACE_IMAGES:
    for _, row in train_df.sample(3, random_state=SEED).iterrows():
        face = open_face(row["path"], row[BOX_COLUMNS])
        fig, ax = plt.subplots(figsize=(3, 3))
        ax.imshow(face)
        ax.set_title(CLASS_NAMES[int(row["label"])])
        ax.axis("off")
        plt.show()

## 14. Prepare the model input

In this cell we read small batches of face images. Each image is RGB and 224 by 224 pixels. Pixel values stay from 0 to 255 because input scaling is already inside the model (TensorFlow, n.d.-b).


In [ ]:
# cell:input
def load_pixels(path, box):
    # Pillow applies the same face crop for image checks and model input.
    path = path.item() if isinstance(path, np.ndarray) else path
    path = path.decode("utf-8") if isinstance(path, bytes) else str(path)
    image = open_face(path, box).resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
    return np.asarray(image, dtype=np.float32)

def read_image(path, box, label):
    image = tf.numpy_function(load_pixels, [path, box], tf.float32)
    image.set_shape((IMAGE_SIZE, IMAGE_SIZE, 3))
    return image, label

def make_dataset(frame, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((
        frame["path"].astype(str).to_numpy(),
        frame[BOX_COLUMNS].to_numpy(dtype=np.float32),
        frame["label"].to_numpy(dtype=np.int32),
    ))
    if training:
        dataset = dataset.shuffle(min(len(frame), 10000), seed=SEED)
    dataset = dataset.map(read_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
images, labels = next(iter(train_ds))
print("Batch shape:", images.shape, "Pixel range:", float(tf.reduce_min(images)),
      "to", float(tf.reduce_max(images)))

## 15. Make small changes to training images

In this cell we make small random changes to training images. We use horizontal flips, brightness and contrast changes. The function is compiled once so it does not create repeated TensorFlow warnings.

In [ ]:
# cell:augmentation
# This function is traced once and reused by the training dataset.
# It avoids repeated Keras Random* conversion and retracing warnings.

@tf.function(
    input_signature=[
        tf.TensorSpec(
            shape=[None, IMAGE_SIZE, IMAGE_SIZE, 3],
            dtype=tf.float32,
        ),
        tf.TensorSpec(
            shape=[None],
            dtype=tf.int32,
        ),
    ],
    reduce_retracing=True,
)
def augment_batch(images, labels):
    images = tf.image.random_flip_left_right(
        images,
        seed=SEED,
    )

    images = tf.image.random_brightness(
        images,
        max_delta=10.0,
        seed=SEED + 1,
    )

    images = tf.image.random_contrast(
        images,
        lower=0.90,
        upper=1.10,
        seed=SEED + 2,
    )

    images = tf.clip_by_value(
        images,
        0.0,
        255.0,
    )

    return images, labels


train_ds = (
    train_ds
    .map(
        augment_batch,
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    .prefetch(tf.data.AUTOTUNE)
)

print(
    "Training augmentation ready: horizontal flip, "
    "brightness and contrast."
)

## 16. Give smaller classes more weight

In this cell we give smaller training classes more weight in the loss. The weights use training data only. We cap them so a very small class does not dominate training.


In [ ]:
# cell:weights
train_counts = train_df["label"].value_counts().sort_index()
weights = np.sqrt(len(train_df) / (N_CLASSES * train_counts.to_numpy()))
weights = np.minimum(weights, 3.0)
weights /= np.average(weights, weights=train_counts.to_numpy())
CLASS_WEIGHTS = {int(label): float(weight) for label, weight in zip(train_counts.index, weights)}
print("Class weights:", CLASS_WEIGHTS)

## 17. Build the model

In this cell we build EfficientNetV2B0. We start with ImageNet weights and add our own output layer. The image-feature layers stay frozen at first. Model source: Tan and Le (2021).


In [ ]:
# cell:model
if not globals().get("GPU_READY", False):
    raise RuntimeError("Pass the GPU test before building the model.")


def build_expression_model(weights="imagenet"):
    backbone = tf.keras.applications.EfficientNetV2B0(
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        include_top=False,
        weights=weights,
        include_preprocessing=True,
    )

    backbone.trainable = False

    inputs = tf.keras.Input(
        shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
        name="face_rgb",
    )

    features = backbone(
        inputs,
        training=False,
    )

    features = tf.keras.layers.GlobalAveragePooling2D()(features)
    features = tf.keras.layers.Dropout(0.35)(features)

    outputs = tf.keras.layers.Dense(
        N_CLASSES,
        activation="softmax",
        dtype="float32",
        name="class_scores",
    )(features)

    network = tf.keras.Model(
        inputs,
        outputs,
        name=TASK + "_model",
    )

    return network, backbone.name


with tf.device("/GPU:0"):
    model, BACKBONE_NAME = build_expression_model(
        weights="imagenet"
    )

model.summary()
print("Number of model parameters:", model.count_params())

## 18. Set up training

In this cell we choose the loss and learning rate. We save the best validation checkpoint and stop early when validation loss stops improving.


In [ ]:
# cell:training_setup
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


@tf.keras.utils.register_keras_serializable(package="EdgeAI")
class ExpressionRecall(tf.keras.metrics.Metric):
    def __init__(
        self,
        class_id=None,
        name="balanced_accuracy",
        **kwargs,
    ):
        super().__init__(
            name=name,
            dtype=tf.float32,
            **kwargs,
        )

        if class_id is not None and class_id not in range(N_CLASSES):
            raise ValueError("class_id is outside the expression class range.")

        self.class_id = class_id

        self.cm = self.add_weight(
            name="counts",
            shape=(N_CLASSES, N_CLASSES),
            initializer="zeros",
            dtype=tf.float32,
        )

    def update_state(
        self,
        y_true,
        y_pred,
        sample_weight=None,
    ):
        # Class weights are used for the loss only.
        # These recall values stay unweighted and easy to interpret.
        truth = tf.cast(
            tf.reshape(y_true, [-1]),
            tf.int32,
        )

        prediction = tf.argmax(
            y_pred,
            axis=-1,
            output_type=tf.int32,
        )

        counts = tf.math.confusion_matrix(
            truth,
            prediction,
            num_classes=N_CLASSES,
            dtype=tf.float32,
        )

        self.cm.assign_add(counts)

    def result(self):
        recalls = tf.math.divide_no_nan(
            tf.linalg.diag_part(self.cm),
            tf.reduce_sum(self.cm, axis=1),
        )

        if self.class_id is None:
            return tf.reduce_mean(recalls)

        return recalls[self.class_id]

    def reset_state(self):
        self.cm.assign(
            tf.zeros_like(self.cm)
        )

    def get_config(self):
        return dict(
            super().get_config(),
            class_id=self.class_id,
        )


RECALL_NAMES = [
    "recall_angry",
    "recall_disgust",
    "recall_fear",
    "recall_happy",
    "recall_sad",
    "recall_surprise",
    "recall_neutral",
]


def compile_model(network, learning_rate):
    metrics = ["accuracy"]

    metrics.extend(
        ExpressionRecall(
            class_id=index,
            name=RECALL_NAMES[index],
        )
        for index in range(N_CLASSES)
    )

    metrics.append(
        ExpressionRecall(
            name="balanced_accuracy"
        )
    )

    network.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=1e-4,
        ),
        loss="sparse_categorical_crossentropy",
        metrics=metrics,
        jit_compile=False,
    )


class PrintExpressionRecall(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        values = logs or {}

        parts = []

        for class_name, metric_name in zip(
            CLASS_NAMES,
            RECALL_NAMES,
        ):
            parts.append(
                f"{class_name}: "
                f"{values.get('val_' + metric_name, float('nan')):.1%}"
            )

        print(
            "Validation recall - "
            + " | ".join(parts)
            + f" | Balanced: "
            f"{values.get('val_balanced_accuracy', float('nan')):.1%}",
            flush=True,
        )


def callbacks(stage):
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    return [
        tf.keras.callbacks.ModelCheckpoint(
            str(
                OUTPUT_DIR
                / f"best_{stage}.weights.h5"
            ),
            monitor="val_balanced_accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=True,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_balanced_accuracy",
            mode="max",
            patience=5,
            restore_best_weights=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_balanced_accuracy",
            mode="max",
            patience=2,
            factor=0.5,
            min_lr=1e-7,
        ),
        tf.keras.callbacks.CSVLogger(
            str(
                OUTPUT_DIR
                / f"{stage}_training.csv"
            )
        ),
        PrintExpressionRecall(),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


compile_model(
    model,
    1e-3,
)

print(
    "Each epoch reports validation recall for all seven "
    "expressions and balanced accuracy."
)
print(
    "The best checkpoint is selected by validation "
    "balanced accuracy."
)

## 19. Train the new output layer

In this cell we train the model's new output layer. The pretrained image-feature layers do not change yet.


In [ ]:
# cell:train_head
if not globals().get("GPU_READY", False):
    raise RuntimeError("GPU check has not passed. Training is stopped.")

started = time.perf_counter()

head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks("head"),
    verbose=2,
)

head_seconds = time.perf_counter() - started

print(
    "First training stage, minutes:",
    round(head_seconds / 60, 2),
)

## 20. Prepare fine-tuning

In this cell we load the best first-stage model. We allow the last third of the feature layers to learn. Batch-normalisation layers stay frozen.


In [ ]:
# cell:unfreeze
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model.load_weights(
    str(
        OUTPUT_DIR
        / "best_head.weights.h5"
    )
)

backbone = model.get_layer(
    BACKBONE_NAME
)

backbone.trainable = True

cutoff = int(
    len(backbone.layers) * 2 / 3
)

for index, layer in enumerate(
    backbone.layers
):
    layer.trainable = (
        index >= cutoff
        and not isinstance(
            layer,
            tf.keras.layers.BatchNormalization,
        )
    )

compile_model(
    model,
    1e-5,
)

print(
    "Trainable feature layers:",
    sum(
        layer.trainable
        for layer in backbone.layers
    ),
)

## 21. Fine-tune the model

In this cell we train again with a smaller learning rate. The output layer and selected feature layers learn together.


In [ ]:
# cell:train_fine
if not globals().get("GPU_READY", False):
    raise RuntimeError("GPU check has not passed. Training is stopped.")

started = time.perf_counter()

fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks("fine"),
    verbose=2,
)

fine_seconds = time.perf_counter() - started

print(
    "Fine-tuning, minutes:",
    round(fine_seconds / 60, 2),
)

## 22. Keep the best model

In this cell we compare the two saved checkpoints. Validation balanced accuracy chooses the model, and validation loss only breaks a tie. The holdout does not choose the winner.

In [ ]:
# cell:select
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

choices = []

for stage in ["head", "fine"]:
    model.load_weights(
        str(
            OUTPUT_DIR
            / f"best_{stage}.weights.h5"
        )
    )

    compile_model(
        model,
        1e-5,
    )

    result = model.evaluate(
        val_ds,
        verbose=0,
        return_dict=True,
    )

    row = {
        "stage": stage,
        "validation_loss": float(result["loss"]),
        "validation_accuracy": float(result["accuracy"]),
        "validation_balanced_accuracy": float(
            result["balanced_accuracy"]
        ),
    }

    for class_name, metric_name in zip(
        CLASS_NAMES,
        RECALL_NAMES,
    ):
        row[
            "validation_" + metric_name
        ] = float(result[metric_name])

    choices.append(row)


selection = (
    pd.DataFrame(choices)
    .sort_values(
        [
            "validation_balanced_accuracy",
            "validation_loss",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(selection)

selection.to_csv(
    OUTPUT_DIR / "checkpoint_selection.csv",
    index=False,
)

BEST_STAGE = str(
    selection.iloc[0]["stage"]
)

model.load_weights(
    str(
        OUTPUT_DIR
        / f"best_{BEST_STAGE}.weights.h5"
    )
)

model.save(
    OUTPUT_DIR / "best_model.h5",
    include_optimizer=False,
)

print("Chosen stage:", BEST_STAGE)

## 23. Plot the training results

In this cell we plot loss, accuracy, balanced accuracy and the validation recall for every expression. The dashed line shows when fine-tuning started.

In [ ]:
# cell:curves
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

history = pd.concat(
    [
        pd.read_csv(
            OUTPUT_DIR / "head_training.csv"
        ),
        pd.read_csv(
            OUTPUT_DIR / "fine_training.csv"
        ),
    ],
    ignore_index=True,
)

history.to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False,
)

fine_start = (
    len(head_history.history["loss"])
    + 0.5
)

for metric in [
    "loss",
    "accuracy",
    "balanced_accuracy",
]:
    fig, ax = plt.subplots(
        figsize=(8, 4)
    )

    ax.plot(
        np.arange(1, len(history) + 1),
        history[metric],
        label="Training",
    )

    ax.plot(
        np.arange(1, len(history) + 1),
        history["val_" + metric],
        label="Validation",
    )

    ax.axvline(
        fine_start,
        linestyle="--",
    )

    ax.set(
        xlabel="Epoch",
        ylabel=metric.replace("_", " ").title(),
        title=metric.replace("_", " ").title()
        + " during training",
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        OUTPUT_DIR
        / f"training_{metric}.png",
        dpi=150,
    )

    plt.show()


# Show validation recall for all seven expressions on one graph.
fig, ax = plt.subplots(
    figsize=(10, 5)
)

for class_name, metric_name in zip(
    CLASS_NAMES,
    RECALL_NAMES,
):
    ax.plot(
        np.arange(1, len(history) + 1),
        history["val_" + metric_name],
        label=class_name,
    )

ax.axvline(
    fine_start,
    linestyle="--",
)

ax.set(
    xlabel="Epoch",
    ylabel="Validation recall",
    title="Validation recall for each expression",
)

ax.legend(
    ncol=2,
)

fig.tight_layout()

fig.savefig(
    OUTPUT_DIR
    / "validation_recall_by_expression.png",
    dpi=150,
)

plt.show()

## 24. Test the model on the holdout

In this cell we test the chosen model on data that did not guide training. We compare it with a simple baseline that always predicts the largest training class. Macro F1 gives each class equal importance.


In [ ]:
# cell:evaluate
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
probabilities = model.predict(test_ds, verbose=0)
if not np.isfinite(probabilities).all():
    raise ValueError("The model produced invalid scores.")
y_true = test_df["label"].to_numpy()
y_pred = probabilities.argmax(axis=1)
majority_class = int(train_df["label"].value_counts().idxmax())
baseline_pred = np.full(len(y_true), majority_class)

def scores(truth, prediction):
    return {"accuracy": float(accuracy_score(truth, prediction)),
            "balanced_accuracy": float(balanced_accuracy_score(truth, prediction)),
            "macro_f1": float(f1_score(truth, prediction, average="macro",
                                       labels=list(range(N_CLASSES)), zero_division=0))}

model_scores = scores(y_true, y_pred)
results = pd.DataFrame([{"model": "Majority-class baseline", **scores(y_true, baseline_pred)},
                        {"model": TASK + " model", **model_scores}])
display(results)
results.to_csv(OUTPUT_DIR / "holdout_results.csv", index=False)
predictions = test_df.copy()
predictions["true_class"] = [CLASS_NAMES[value] for value in y_true]
predictions["predicted_class"] = [CLASS_NAMES[value] for value in y_pred]
predictions["model_score"] = probabilities.max(axis=1)
predictions["correct"] = y_true == y_pred
predictions.to_csv(OUTPUT_DIR / "holdout_predictions.csv", index=False)

## 25. Check each class

In this cell we show results for every class. The confusion matrix shows the real labels in rows and predictions in columns.


In [ ]:
# cell:class_report
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report = pd.DataFrame(classification_report(y_true, y_pred, labels=list(range(N_CLASSES)),
                      target_names=CLASS_NAMES, output_dict=True, zero_division=0)).T
display(report)
report.to_csv(OUTPUT_DIR / "classification_report.csv")
cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUTPUT_DIR / "confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False,
                                                          xticks_rotation=45)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()

## 26. Check happy and sad faces

In this cell we check the two expressions named in the course experiment. The model still chooses from all seven classes. We do not make the test easier by forcing a happy-or-sad answer.


In [ ]:
# cell:task_check
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HAPPY_LABEL = CLASS_NAMES.index("Happy")
SAD_LABEL = CLASS_NAMES.index("Sad")

if (HAPPY_LABEL, SAD_LABEL) != (3, 4):
    raise RuntimeError(
        "The Happy/Sad class order changed. "
        "Check CLASS_NAMES before evaluating the course task."
    )

happy_sad = predictions.loc[
    predictions["label"].isin(
        [
            HAPPY_LABEL,
            SAD_LABEL,
        ]
    )
].copy()

happy_sad_results = (
    happy_sad
    .groupby("true_class")
    .agg(
        images=("correct", "size"),
        recall=("correct", "mean"),
    )
)

display(happy_sad_results)

happy_sad_results.to_csv(
    OUTPUT_DIR
    / "happy_sad_results.csv"
)

print(
    "Happy and Sad are evaluated as part of the full "
    "seven-class model. Predictions are not forced to only two choices."
)

## 27. Look at the mistakes

In this cell we show mistakes with high model scores. A high score is not a guarantee that the prediction is correct.


In [ ]:
# cell:errors
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
mistakes = predictions.loc[~predictions["correct"]].sort_values("model_score", ascending=False)
display(mistakes[["true_class", "predicted_class", "model_score"]].head(10))
mistakes.to_csv(OUTPUT_DIR / "mistakes.csv", index=False)
if SHOW_FACE_IMAGES:
    for _, row in mistakes.head(3).iterrows():
        fig, ax = plt.subplots(figsize=(3, 3))
        ax.imshow(open_face(row["path"], row[BOX_COLUMNS]))
        ax.set_title(f"Label: {row['true_class']}\nPrediction: {row['predicted_class']}")
        ax.axis("off")
        plt.show()

## 28. Export the normal Android model

In this cell we rebuild the chosen network in float32 on the CPU and convert it to TensorFlow Lite. This gives the conversion check a clean reference that does not depend on GPU mixed precision.

In [ ]:
# cell:fp32
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Training uses mixed_float16 on the GPU.
# For deployment checking we build the same architecture in float32 on CPU.
training_policy = tf.keras.mixed_precision.global_policy()

tf.keras.mixed_precision.set_global_policy(
    "float32"
)

try:
    with tf.device("/CPU:0"):
        cpu_reference, _ = build_expression_model(
            weights=None
        )

        cpu_reference.set_weights(
            model.get_weights()
        )

        cpu_reference.trainable = False

        cpu_reference(
            tf.zeros(
                (
                    1,
                    IMAGE_SIZE,
                    IMAGE_SIZE,
                    3,
                ),
                dtype=tf.float32,
            ),
            training=False,
        )

        converter = tf.lite.TFLiteConverter.from_keras_model(
            cpu_reference
        )

        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS
        ]

        fp32_bytes = converter.convert()

finally:
    tf.keras.mixed_precision.set_global_policy(
        training_policy
    )


FP32_PATH = (
    OUTPUT_DIR
    / f"{TASK}_fp32.tflite"
)

FP32_PATH.write_bytes(
    fp32_bytes
)

print("Saved:", FP32_PATH)
print(
    "Size in MB:",
    round(
        FP32_PATH.stat().st_size / 1e6,
        2,
    ),
)

## 29. Try a smaller Android model

In this cell we try INT8 conversion. It uses smaller numbers and may change accuracy. Only training images set the number ranges. If conversion fails, we keep the normal model and record the error.


In [ ]:
# cell:int8
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
calibration_df = pd.concat([group.sample(min(40, len(group)), random_state=SEED)
                           for _, group in train_df.groupby("label")])
calibration_df.to_csv(OUTPUT_DIR / "calibration_images.csv", index=False)

def representative_images():
    for images, _ in make_dataset(calibration_df):
        for image in images.numpy():
            yield [image[None].astype(np.float32)]

INT8_PATH = None
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(cpu_reference)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_images
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    converted = converter.convert()
    INT8_PATH = OUTPUT_DIR / f"{TASK}_int8.tflite"
    INT8_PATH.write_bytes(converted)
    print("Saved:", INT8_PATH)
except Exception as error:
    print("INT8 conversion failed. The FP32 model is still available.")
    print(type(error).__name__ + ":", error)
    (OUTPUT_DIR / "int8_error.txt").write_text(str(error), encoding="utf-8")

## 30. Check the converted models

In this cell we compare TensorFlow Lite with a separate float32 Keras model on the CPU. The check uses the same number of images from every expression class and compares prediction agreement, balanced accuracy and class recall as well as score differences.

In [ ]:
# cell:mobile_check
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def balanced_check_sample(
    frame,
    per_class,
    seed,
):
    parts = []

    for label, group in frame.groupby(
        "label"
    ):
        count = min(
            per_class,
            len(group),
        )

        parts.append(
            group.sample(
                count,
                random_state=seed + int(label),
            )
        )

    result = pd.concat(
        parts,
        ignore_index=True,
    )

    if set(result["label"]) != set(range(N_CLASSES)):
        raise ValueError(
            "The conversion sample must contain all seven expressions."
        )

    return result


def probability_check(
    values,
    rows,
    tolerance=1e-4,
):
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    if (
        values.shape
        != (rows, N_CLASSES)
        or not np.isfinite(values).all()
    ):
        raise ValueError(
            "Wrong prediction shape or invalid model scores."
        )

    if (
        (values < -tolerance).any()
        or (values > 1 + tolerance).any()
    ):
        raise ValueError(
            "Model scores are outside the probability range."
        )

    if not np.allclose(
        values.sum(axis=1),
        1,
        rtol=0,
        atol=tolerance,
    ):
        raise ValueError(
            "Class probabilities do not sum to one."
        )

    return values


def class_recalls(
    truth,
    prediction,
):
    truth = np.asarray(truth)
    prediction = np.asarray(prediction)

    if set(np.unique(truth)) != set(range(N_CLASSES)):
        raise ValueError(
            "The check sample must contain all seven expression classes."
        )

    return np.array(
        [
            np.mean(
                prediction[truth == class_id]
                == class_id
            )
            for class_id in range(N_CLASSES)
        ],
        dtype=np.float32,
    )


def conversion_report(
    truth,
    reference,
    candidate,
    limits,
    quantized=False,
):
    truth = np.asarray(truth)

    reference = probability_check(
        reference,
        len(truth),
    )

    candidate = probability_check(
        candidate,
        len(truth),
        tolerance=0.03 if quantized else 1e-4,
    )

    difference = np.abs(
        reference - candidate
    )

    reference_labels = reference.argmax(
        axis=1
    )

    candidate_labels = candidate.argmax(
        axis=1
    )

    reference_recall = class_recalls(
        truth,
        reference_labels,
    )

    candidate_recall = class_recalls(
        truth,
        candidate_labels,
    )

    result = {
        "max_abs_difference": float(
            difference.max()
        ),
        "mean_abs_difference": float(
            difference.mean()
        ),
        "p99_abs_difference": float(
            np.quantile(
                difference,
                0.99,
            )
        ),
        "prediction_agreement": float(
            np.mean(
                reference_labels
                == candidate_labels
            )
        ),
        "accuracy": float(
            np.mean(
                truth
                == candidate_labels
            )
        ),
        "balanced_accuracy": float(
            candidate_recall.mean()
        ),
        "balanced_accuracy_drop": float(
            reference_recall.mean()
            - candidate_recall.mean()
        ),
        "largest_class_recall_drop": float(
            np.max(
                reference_recall
                - candidate_recall
            )
        ),
    }

    for class_name, value in zip(
        CLASS_NAMES,
        candidate_recall,
    ):
        result[
            "recall_"
            + class_name.lower()
        ] = float(value)

    reasons = []

    checks = [
        (
            "max_abs_difference",
            "max_abs",
        ),
        (
            "mean_abs_difference",
            "mean_abs",
        ),
        (
            "balanced_accuracy_drop",
            "max_balanced_accuracy_drop",
        ),
        (
            "largest_class_recall_drop",
            "max_class_recall_drop",
        ),
    ]

    for key, limit_key in checks:
        if (
            result[key]
            > limits[limit_key]
            + 1e-12
        ):
            reasons.append(
                key
                + " exceeds the set limit"
            )

    if (
        result["prediction_agreement"]
        + 1e-12
        < limits["min_agreement"]
    ):
        reasons.append(
            "prediction agreement is below the set limit"
        )

    result["passed"] = not reasons
    result["reason"] = (
        "; ".join(reasons)
        or "All checks passed"
    )

    return result


def encode_input(
    value,
    detail,
):
    value = np.asarray(
        value,
        dtype=np.float32,
    )

    if not np.isfinite(value).all():
        raise ValueError(
            "Input contains invalid pixels."
        )

    if np.issubdtype(
        detail["dtype"],
        np.integer,
    ):
        scale, zero = detail[
            "quantization"
        ]

        if (
            not np.isfinite(scale)
            or scale <= 0
        ):
            raise ValueError(
                "Invalid input quantization scale."
            )

        limits = np.iinfo(
            detail["dtype"]
        )

        value = np.clip(
            np.rint(
                value / scale
                + zero
            ),
            limits.min,
            limits.max,
        )

    return value.astype(
        detail["dtype"]
    )


def decode_output(
    value,
    detail,
):
    value = np.asarray(value)

    if np.issubdtype(
        detail["dtype"],
        np.integer,
    ):
        scale, zero = detail[
            "quantization"
        ]

        if (
            not np.isfinite(scale)
            or scale <= 0
        ):
            raise ValueError(
                "Invalid output quantization scale."
            )

        value = (
            value.astype(np.float32)
            - zero
        ) * scale

    return value.astype(
        np.float32
    )


def open_lite(path):
    interpreter = tf.lite.Interpreter(
        model_path=str(path),
        num_threads=4,
    )

    interpreter.allocate_tensors()

    inputs = interpreter.get_input_details()
    outputs = interpreter.get_output_details()

    if len(inputs) != 1 or len(outputs) != 1:
        raise ValueError(
            "Expected one image input and one expression output."
        )

    expected_input = [
        1,
        IMAGE_SIZE,
        IMAGE_SIZE,
        3,
    ]

    expected_output = [
        1,
        N_CLASSES,
    ]

    if (
        list(inputs[0]["shape"])
        != expected_input
        or list(outputs[0]["shape"])
        != expected_output
    ):
        raise ValueError(
            "Unexpected TensorFlow Lite input/output shape."
        )

    return (
        interpreter,
        inputs[0],
        outputs[0],
    )


def lite_predictions(
    path,
    frame,
):
    interpreter, input_info, output_info = open_lite(
        path
    )

    values = []

    for images, _ in make_dataset(frame):
        for image in images.numpy():
            interpreter.set_tensor(
                input_info["index"],
                encode_input(
                    image[None],
                    input_info,
                ),
            )

            interpreter.invoke()

            values.append(
                decode_output(
                    interpreter.get_tensor(
                        output_info["index"]
                    )[0],
                    output_info,
                )
            )

    return np.asarray(
        values,
        dtype=np.float32,
    )


def cpu_predictions(frame):
    values = []

    for images, _ in make_dataset(frame):
        with tf.device("/CPU:0"):
            values.append(
                cpu_reference(
                    images,
                    training=False,
                ).numpy()
            )

    return np.concatenate(
        values
    )


validation_sample = balanced_check_sample(
    val_df,
    MOBILE_IMAGES_PER_CLASS,
    SEED,
)

validation_sample.to_csv(
    OUTPUT_DIR
    / "conversion_validation_images.csv",
    index=False,
)

truth = validation_sample[
    "label"
].to_numpy()

keras_scores = cpu_predictions(
    validation_sample
)

fp32_scores = lite_predictions(
    FP32_PATH,
    validation_sample,
)

fp32_report = conversion_report(
    truth,
    keras_scores,
    fp32_scores,
    FP32_LIMITS,
)

# GPU/CPU drift is diagnostic only.
gpu_scores = model.predict(
    make_dataset(validation_sample),
    verbose=0,
)

print(
    "Largest GPU-Keras / CPU-Keras difference:",
    float(
        np.max(
            np.abs(
                gpu_scores
                - keras_scores
            )
        )
    ),
)

print(
    "Largest CPU-Keras / FP32-Lite difference:",
    fp32_report["max_abs_difference"],
)

print(
    "FP32 class-prediction agreement:",
    f"{fp32_report['prediction_agreement']:.2%}",
)

format_rows = [
    dict(
        format="FP32",
        size_MB=(
            FP32_PATH.stat().st_size
            / 1e6
        ),
        **fp32_report,
    )
]

mobile_models = {
    "FP32": FP32_PATH
}

CHOSEN_FORMAT = "FP32"

if INT8_PATH is not None:
    try:
        int8_scores = lite_predictions(
            INT8_PATH,
            validation_sample,
        )

        int8_report = conversion_report(
            truth,
            fp32_scores,
            int8_scores,
            INT8_LIMITS,
            quantized=True,
        )

        format_rows.append(
            dict(
                format="INT8",
                size_MB=(
                    INT8_PATH.stat().st_size
                    / 1e6
                ),
                **int8_report,
            )
        )

        if (
            fp32_report["passed"]
            and int8_report["passed"]
        ):
            mobile_models[
                "INT8"
            ] = INT8_PATH

            if (
                INT8_PATH.stat().st_size
                < FP32_PATH.stat().st_size
            ):
                CHOSEN_FORMAT = "INT8"

        else:
            print(
                "INT8 is not selected:",
                int8_report["reason"],
            )

    except Exception as error:
        print(
            "INT8 checking failed; "
            "keeping FP32:",
            error,
        )

        (
            OUTPUT_DIR
            / "int8_validation_error.txt"
        ).write_text(
            str(error),
            encoding="utf-8",
        )


format_table = (
    pd.DataFrame(format_rows)
    .set_index("format")
)

display(format_table)

format_table.to_csv(
    OUTPUT_DIR
    / "format_validation.csv"
)

(
    OUTPUT_DIR
    / "conversion_limits.json"
).write_text(
    json.dumps(
        {
            "FP32": FP32_LIMITS,
            "INT8": INT8_LIMITS,
            "sample_images_per_class": MOBILE_IMAGES_PER_CLASS,
            "sample_method": "equal number from every expression class",
            "fp32_reference": (
                "Separate float32 Keras model on CPU "
                "with identical weights and preprocessing"
            ),
        },
        indent=2,
    ),
    encoding="utf-8",
)

if not fp32_report["passed"]:
    raise RuntimeError(
        "FP32 conversion failed the declared checks. "
        "See format_validation.csv: "
        + fp32_report["reason"]
    )

print(
    "Candidate Android format:",
    CHOSEN_FORMAT,
)

## 31. Compare accuracy, size and computer speed

In this cell we compare all formats on the same holdout sample. We measure inference time on this computer's CPU. We still need separate measurements on the real Android phone.


In [ ]:
# cell:comparison
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def computer_latency(path, sample):
    interpreter = tf.lite.Interpreter(model_path=str(path), num_threads=4)
    interpreter.allocate_tensors()
    details = interpreter.get_input_details()[0]
    value = sample[None].astype(np.float32)
    if np.issubdtype(details["dtype"], np.integer):
        scale, zero = details["quantization"]
        limits = np.iinfo(details["dtype"])
        value = np.clip(np.rint(value / scale + zero), limits.min, limits.max)
    interpreter.set_tensor(details["index"], value.astype(details["dtype"]))
    for _ in range(10):
        interpreter.invoke()
    times = []
    for _ in range(50):
        started = time.perf_counter()
        interpreter.invoke()
        times.append((time.perf_counter() - started) * 1000)
    return float(np.median(times)), float(np.percentile(times, 95))

comparison_df = balanced_check_sample(
    test_df,
    MOBILE_IMAGES_PER_CLASS,
    SEED,
)
comparison_df.to_csv(OUTPUT_DIR / "mobile_comparison_images.csv", index=False)
comparison_labels = comparison_df["label"].to_numpy()
keras_comparison = cpu_predictions(comparison_df)
comparison_rows = [{"format": "Keras", "images": len(comparison_df),
                    "accuracy": accuracy_score(comparison_labels, keras_comparison.argmax(axis=1)),
                    "macro_f1": f1_score(comparison_labels, keras_comparison.argmax(axis=1),
                        average="macro", labels=list(range(N_CLASSES)), zero_division=0),
                    "size_MB": (OUTPUT_DIR / "best_model.h5").stat().st_size / 1e6,
                    "computer_CPU_median_ms": np.nan, "computer_CPU_p95_ms": np.nan}]
sample, _ = next(iter(make_dataset(validation_sample.head(1))))
for name, path in mobile_models.items():
    scores = lite_predictions(path, comparison_df)
    median_ms, p95_ms = computer_latency(path, sample[0].numpy())
    comparison_rows.append({"format": name, "images": len(comparison_df),
        "accuracy": accuracy_score(comparison_labels, scores.argmax(axis=1)),
        "macro_f1": f1_score(comparison_labels, scores.argmax(axis=1), average="macro",
                             labels=list(range(N_CLASSES)), zero_division=0),
        "size_MB": path.stat().st_size / 1e6,
        "computer_CPU_median_ms": median_ms, "computer_CPU_p95_ms": p95_ms})
comparison = pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
print("Timing: this computer's CPU, 4 threads, 10 warm-ups, 50 runs.")
print("Image loading, cropping and resizing are not included. These are NOT phone timings.")

## 32. Save the Android model and its input rules

In this cell we save the chosen TensorFlow Lite model, the label order and the input rules. Android must use the same RGB face crop and image size.

In [ ]:
# cell:save
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHOSEN_PATH = OUTPUT_DIR / f"{TASK}_android.tflite"
shutil.copyfile(mobile_models[CHOSEN_FORMAT], CHOSEN_PATH)
(OUTPUT_DIR / "labels.txt").write_text("\n".join(CLASS_NAMES) + "\n", encoding="utf-8")
interpreter = tf.lite.Interpreter(model_path=str(CHOSEN_PATH))
interpreter.allocate_tensors()

def tensor_rules(details):
    return {"shape": details["shape"].tolist(), "dtype": np.dtype(details["dtype"]).name,
            "scale": float(details["quantization"][0]), "zero_point": int(details["quantization"][1])}

settings = {
    "course": "DLBAIPEAI", "task": "Task 2", "model_task": TASK,
    "architecture": "EfficientNetV2B0", "dataset": "ExpW",
    "label_kaggle_handle": LABEL_HANDLE, "image_kaggle_handle": IMAGE_HANDLE, "seed": SEED, "class_names": CLASS_NAMES,
    "input": tensor_rules(interpreter.get_input_details()[0]),
    "output": tensor_rules(interpreter.get_output_details()[0]),
    "image_rule": "Crop face, convert to RGB, resize to 224x224 using bilinear interpolation.",
    "pixel_rule": "Float pixels 0-255. Scaling is already inside the model. Do not divide by 255.",
    "integer_rule": "For integer tensors use the saved scale and zero point, with rounding and clipping.",
    "includes_face_detector": False, "measured_on_phone": False,
    "split_sizes": {name: len(frame) for name, frame in frames.items()},
    "chosen_stage": BEST_STAGE, "chosen_format": CHOSEN_FORMAT,
    "training_seconds": head_seconds + fine_seconds,
    "gpu_name": GPU_NAME, "gpu_forward_backward_test_passed": GPU_READY,
    "holdout_scores": model_scores,
    "split_method": "StratifiedGroupKFold(10): fold 0 holdout, fold 1 validation, folds 2-9 training; grouped by original image path",
    "checkpoint_rule": "maximum validation balanced accuracy; validation loss breaks a tie",
    "conversion_sample_images_per_class": MOBILE_IMAGES_PER_CLASS,
    "versions": {name: metadata.version(name) for name in packages},
    "system": platform.platform(), "tensorflow_devices": [str(device) for device in gpus],
}
(OUTPUT_DIR / "model_settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
print("Android model:", CHOSEN_PATH)
print("The training input starts with a cropped face. The app must do this crop too.")

## 33. Summarise this run

In this cell we summarise the measured results. The notebook does not create results before training. The Android app and the required 20-image device experiment still need to be completed.


In [ ]:
# cell:summary
display(Markdown(
    f"### Measured results\n\n"
    f"The model was trained on **{len(train_df):,} images**. "
    f"The validation set had **{len(val_df):,} images**. "
    f"The final holdout had **{len(test_df):,} images**.\n\n"
    f"Holdout accuracy was **{model_scores['accuracy']:.2%}**. "
    f"Macro F1 was **{model_scores['macro_f1']:.3f}**. "
    f"The selected checkpoint was **{BEST_STAGE}**, and the candidate phone format was **{CHOSEN_FORMAT}**.\n\n"
    f"Review the class report and mistakes above. Exact duplicates were checked, "
    f"but near-duplicates and all repeated people were not ruled out. "
    f"ExpW does not provide a fixed train/test split here, so this notebook uses a fixed seeded group-aware split. "
    f"Faces from the same original photograph cannot cross training, validation and holdout sets. "
    f"These computer tests do not replace on-phone tests."
))

### What remains for Task 2

The final phone app must recognise age, gender and expression on the device. Test it with the 20 device-captured images required in the task brief. Do not use those images for training or model selection. Record consent and protect personal images. Include real phone measurements in the final report (IU International University, n.d., pp. 3-4).

The source code, critical review of the results and AI-use statement still belong in the course submission. A working model alone does not guarantee a particular grade.


### Sources

IU International University. (n.d.). *DLBAIPEAI - Project Edge AI: Task - Project report* (Task 2, pp. 3-4). Course document supplied with this project.

Zhang, Z., Luo, P., Loy, C. C., & Tang, X. (2018). From facial expression recognition to interpersonal relation prediction. *International Journal of Computer Vision, 126*(5), 550-569. https://doi.org/10.1007/s11263-017-1055-1

Kaggle. (n.d.). *Expression in-the-Wild (ExpW) Dataset*. Label dataset mirror used by this notebook: https://www.kaggle.com/datasets/nguhaduong/expression-in-the-wild-expw-dataset

Kaggle. (n.d.). *origin expw*. Original image mirror used by this notebook: https://www.kaggle.com/datasets/minhtmnguyntrn/origin-expw

Tan, M., & Le, Q. V. (2021). EfficientNetV2: Smaller models and faster training. *Proceedings of Machine Learning Research, 139*, 10096-10106. https://proceedings.mlr.press/v139/tan21a.html

TensorFlow. (n.d.). *Install TensorFlow with pip*. https://www.tensorflow.org/install/pip

TensorFlow. (n.d.). *EfficientNetV2B0*. https://www.tensorflow.org/api_docs/python/tf/keras/applications/EfficientNetV2B0

TensorFlow. (n.d.). *Post-training integer quantization*. https://www.tensorflow.org/lite/performance/post_training_integer_quant

All result tables and graphs in this notebook are produced from the loaded data when the cells run.

## 34. Make the notebook PDF

In this cell we automatically create a PDF of the saved notes, code and recorded outputs. Save any changes to the notes first. Training is not run again. No LaTeX, browser installer or helper file is needed.


In [ ]:
# cell:pdf
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

import base64
import textwrap
import copy
import re
import io

from html import escape

import nbformat

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import (
    getSampleStyleSheet,
    ParagraphStyle,
)
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Preformatted,
    Spacer,
    Image as PDFImage,
)


# Get the current notebook directly from Google Colab.
try:
    from google.colab import _message

    response = _message.blocking_request(
        "get_ipynb",
        timeout_sec=30,
    )

    notebook_data = response.get(
        "ipynb"
    )

    if notebook_data is None:
        raise RuntimeError(
            "Colab did not return the current notebook."
        )

    notebook = nbformat.from_dict(
        notebook_data
    )

except Exception as error:
    raise RuntimeError(
        "Could not read the current Colab notebook."
    ) from error


# Colab can return cell source as either a string
# or a list of strings. Make every source one string.
for cell in notebook.cells:

    source = cell.get(
        "source",
        "",
    )

    if isinstance(
        source,
        list,
    ):
        source = "".join(
            source
        )

    elif not isinstance(
        source,
        str,
    ):
        source = str(
            source
        )

    cell["source"] = source

    if cell.cell_type == "code":

        key = (
            source.splitlines()[0].strip()
            if source
            else ""
        )

        cell.outputs = []
        cell.execution_count = None

        if key in pdf_recorder.records:

            record = copy.deepcopy(
                pdf_recorder.records[key]
            )

            cell.outputs = nbformat.from_dict(
                {
                    "outputs": record.get(
                        "outputs",
                        [],
                    )
                }
            ).outputs

            cell.execution_count = record.get(
                "execution_count"
            )


# PDF styles.
styles = getSampleStyleSheet()

styles.add(
    ParagraphStyle(
        "Notes",
        fontName="Helvetica",
        fontSize=10,
        leading=14,
        spaceAfter=8,
    )
)

styles.add(
    ParagraphStyle(
        "NotebookCode",
        fontName="Courier",
        fontSize=8,
        leading=10.5,
        spaceAfter=7,
    )
)

styles["Heading1"].fontSize = 18
styles["Heading1"].leading = 22

styles["Heading2"].fontSize = 12
styles["Heading2"].leading = 16

story = []


def plain(value):

    if isinstance(
        value,
        list,
    ):
        text = "".join(
            value
        )

    else:
        text = str(
            value
        )

    # Remove terminal colour codes.
    text = re.sub(
        r"\x1b\[[0-?]*[ -/]*[@-~]",
        "",
        text,
    )

    replacements = {
        "–": "-",
        "—": "-",
        "→": "->",
        "’": "'",
        "“": '"',
        "”": '"',
    }

    for old, new in replacements.items():
        text = text.replace(
            old,
            new,
        )

    return (
        text
        .encode(
            "latin-1",
            errors="replace",
        )
        .decode(
            "latin-1"
        )
    )


def inline(text):

    text = escape(
        plain(text)
    )

    text = re.sub(
        r"\[([^\]]+)\]\((https?://[^\s)]+)\)",
        r'<link href="\2">\1</link>',
        text,
    )

    text = re.sub(
        r"\*\*(.+?)\*\*",
        r"<b>\1</b>",
        text,
    )

    text = re.sub(
        r"`([^`]+)`",
        r'<font name="Courier">\1</font>',
        text,
    )

    return text


def add_code(text):

    lines = []

    for line in plain(
        text
    ).splitlines():

        wrapped = textwrap.wrap(
            line.expandtabs(4),
            98,
            replace_whitespace=False,
            drop_whitespace=False,
            break_on_hyphens=False,
        )

        lines.extend(
            wrapped or [""]
        )

    for start in range(
        0,
        len(lines),
        22,
    ):

        story.append(
            Preformatted(
                "\n".join(
                    lines[
                        start:start + 22
                    ]
                ),
                styles["NotebookCode"],
            )
        )


def add_notes(text):

    for paragraph in plain(
        text
    ).split(
        "\n\n"
    ):

        paragraph = paragraph.strip()

        if not paragraph:
            continue

        heading = re.match(
            r"^(#{1,3})\s+(.*)$",
            paragraph,
            re.S,
        )

        if heading:

            style = styles[
                "Heading1"
                if len(
                    heading[1]
                ) == 1
                else "Heading2"
            ]

            story.append(
                Paragraph(
                    inline(
                        heading[2]
                    ),
                    style,
                )
            )

        else:

            story.append(
                Paragraph(
                    inline(
                        paragraph
                    ).replace(
                        "\n",
                        "<br/>",
                    ),
                    styles["Notes"],
                )
            )


# Add every notebook cell to the PDF.
for number, cell in enumerate(
    notebook.cells,
    1,
):

    if cell.cell_type == "markdown":

        add_notes(
            cell.source
        )

    elif cell.cell_type == "code":

        status = (
            "recorded"
            if cell.execution_count is not None
            else "output not recorded"
        )

        story.append(
            Paragraph(
                f"Code cell {number} - {status}",
                styles["Heading3"],
            )
        )

        add_code(
            cell.source
        )

        for output in cell.outputs:

            if output.get(
                "output_type"
            ) == "stream":

                add_code(
                    output.get(
                        "text",
                        "",
                    )
                )

            else:

                data = output.get(
                    "data",
                    {},
                )

                if "image/png" in data:

                    encoded = data[
                        "image/png"
                    ]

                    if isinstance(
                        encoded,
                        list,
                    ):
                        encoded = "".join(
                            encoded
                        )

                    image = PDFImage(
                        io.BytesIO(
                            base64.b64decode(
                                encoded
                            )
                        )
                    )

                    scale = min(
                        500 / image.imageWidth,
                        330 / image.imageHeight,
                        1.0,
                    )

                    image.drawWidth *= scale
                    image.drawHeight *= scale

                    story.append(
                        image
                    )

                    story.append(
                        Spacer(
                            1,
                            8,
                        )
                    )

                elif "text/markdown" in data:

                    add_notes(
                        data[
                            "text/markdown"
                        ]
                    )

                elif "text/plain" in data:

                    add_code(
                        data[
                            "text/plain"
                        ]
                    )


# Save the PDF.
PDF_PATH = (
    OUTPUT_DIR
    / (
        NOTEBOOK_NAME
        + ".pdf"
    )
)


def page_number(
    canvas,
    document,
):

    canvas.setFont(
        "Helvetica",
        8,
    )

    canvas.drawString(
        40,
        22,
        "DLBAIPEAI - Task 2 - "
        + TASK.title()
        + " model",
    )

    canvas.drawRightString(
        A4[0] - 40,
        22,
        f"Page {document.page}",
    )


SimpleDocTemplate(
    str(PDF_PATH),
    pagesize=A4,
    leftMargin=40,
    rightMargin=40,
    topMargin=38,
    bottomMargin=38,
    title=NOTEBOOK_NAME,
    author="W. Pretorius",
).build(
    story,
    onFirstPage=page_number,
    onLaterPages=page_number,
)


# Save an executed notebook copy too.
EXECUTED_PATH = (
    OUTPUT_DIR
    / (
        NOTEBOOK_NAME
        + "_executed.ipynb"
    )
)

nbformat.write(
    notebook,
    EXECUTED_PATH,
)

print(
    "Notebook PDF saved:",
    PDF_PATH,
)

print(
    "Executed notebook saved:",
    EXECUTED_PATH,
)

## Download the Colab results

In this cell we put the trained models, result tables, graphs and PDF into one ZIP file and download it to the computer.


In [ ]:
# cell:colab_download
import shutil
from google.colab import files as colab_files

archive_base = PROJECT_DIR / f"expression_results_{OUTPUT_DIR.name}"

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Created:", archive_path)
colab_files.download(archive_path)